In [ ]:
# ============================================================
# IMPROVED LUNG SOUND ANALYSIS - Key Optimizations
# ============================================================
import os
import random
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Conv1D, MaxPooling1D, Flatten,
    BatchNormalization, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import regularizers, layers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# ============================================================
# Global Params
# ============================================================
SEGMENT_SEC = 2.0  # ✨ REDUCED from 3.0 for better temporal resolution
SR = 22050
N_MFCC = 52
NUM_CLASSES = 3
EPOCHS = 120
BATCH_SIZE = 32
USE_CV = True  # ✨ NEW: Use cross-validation
N_SPLITS = 5   # ✨ NEW: 5-fold cross-validation

# ============================================================
# Audio Utils
# ============================================================
def segment_audio(audio, sr):
    """✨ IMPROVED: 66% overlap for more data samples"""
    win = int(SEGMENT_SEC * sr)
    hop = win // 3  # 66% overlap instead of 50%
    return [audio[i:i + win] for i in range(0, len(audio) - win, hop)]

def augment_noise(audio):
    return audio + 0.005 * np.random.randn(len(audio))

def augment_pitch(audio, sr):
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-2, 2))

def augment_speed(audio):
    return librosa.effects.time_stretch(audio, rate=random.uniform(0.9, 1.1))

def apply_augmentation(audio, sr):
    return random.choice([
        lambda x: augment_noise(x),
        lambda x: augment_pitch(x, sr),
        lambda x: augment_speed(x)
    ])(audio)

# ============================================================
# Feature Extraction - ENHANCED
# ============================================================
def extract_mfcc_seq(audio, sr=SR, max_len=130):
    """MFCC sequence for CNN"""
    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr,
        n_mfcc=N_MFCC,
        n_fft=2048,
        hop_length=512
    )

    mfcc = mfcc[:, :max_len]
    if mfcc.shape[1] < max_len:
        mfcc = np.pad(mfcc, ((0,0),(0, max_len - mfcc.shape[1])))

    return mfcc.T

def extract_mfcc_stats(audio, sr=SR):
    """MFCC statistics (baseline)"""
    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr,
        n_mfcc=N_MFCC,
        n_fft=2048,
        hop_length=512
    )

    delta = librosa.feature.delta(mfcc)

    feats = np.hstack([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.min(axis=1),
        mfcc.max(axis=1),
        delta.mean(axis=1),
        delta.std(axis=1),
    ])

    return feats

def extract_advanced_features(audio, sr=SR):
    """✨ NEW: Enhanced feature extraction for better discrimination"""
    # MFCC
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    mfcc_features = np.hstack([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        mfcc.min(axis=1),
        mfcc.max(axis=1),
    ])
    
    # Delta MFCC
    delta = librosa.feature.delta(mfcc)
    delta_features = np.hstack([
        delta.mean(axis=1),
        delta.std(axis=1),
    ])
    
    # Mel-spectrogram energy
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128)
    mel_features = np.hstack([
        mel_spec.mean(axis=1)[:20],  # Take first 20 dims
        mel_spec.std(axis=1)[:20],
    ])
    
    # Zero crossing rate (texture)
    zcr = librosa.feature.zero_crossing_rate(audio)[0]
    zcr_features = np.array([zcr.mean(), zcr.std()])
    
    # Spectral features
    spec_cent = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    spec_roll = librosa.feature.spectral_rolloff(y=audio, sr=sr)[0]
    spectral_features = np.array([
        spec_cent.mean(), spec_cent.std(),
        spec_roll.mean(), spec_roll.std()
    ])
    
    # RMS energy (intensity variation - important for respiratory patterns)
    rms = librosa.feature.rms(y=audio)[0]
    rms_features = np.array([rms.mean(), rms.std()])
    
    # Combine all
    return np.hstack([
        mfcc_features, delta_features, mel_features,
        zcr_features, spectral_features, rms_features
    ])

def augment_spectrogram(mfcc, freq_mask_param=15, time_mask_param=20):
    """✨ NEW: SpecAugment on MFCC for better regularization"""
    mfcc = mfcc.copy()
    
    # Frequency masking
    f = np.random.randint(0, freq_mask_param)
    if f > 0 and mfcc.shape[0] > f:
        f0 = np.random.randint(0, mfcc.shape[0] - f)
        mfcc[f0:f0+f, :] = 0
    
    # Time masking
    t = np.random.randint(0, time_mask_param)
    if t > 0 and mfcc.shape[1] > t:
        t0 = np.random.randint(0, mfcc.shape[1] - t)
        mfcc[:, t0:t0+t] = 0
    
    return mfcc

# ============================================================
# Dataset - ENHANCED
# ============================================================
class_folders = {
    "Asthma": "./Datasets/Asthma",
    "COPD": "./Datasets/COPD4",
    "Healthy": "./Datasets/Healthy"
}

X_cnn, X_flat_basic, X_flat_adv, y = [], [], [], []

for label, folder in class_folders.items():
    for file in os.listdir(folder):
        audio, sr = librosa.load(os.path.join(folder, file), sr=SR)
        segments = segment_audio(audio, sr)

        for seg in segments:
            # Original segment
            X_cnn.append(extract_mfcc_seq(seg))
            X_flat_basic.append(extract_mfcc_stats(seg))
            X_flat_adv.append(extract_advanced_features(seg))
            y.append(label)

            # ✨ IMPROVED: More augmentations per segment
            for _ in range(2):
                seg_aug = apply_augmentation(seg, sr)
                mfcc_seq = extract_mfcc_seq(seg_aug)
                
                # Apply SpecAugment
                mfcc_seq = augment_spectrogram(mfcc_seq.T).T
                
                X_cnn.append(mfcc_seq)
                X_flat_basic.append(extract_mfcc_stats(seg_aug))
                X_flat_adv.append(extract_advanced_features(seg_aug))
                y.append(label)

X_cnn = pad_sequences(X_cnn, padding="post", dtype="float32")
X_flat_basic = np.array(X_flat_basic, dtype="float32")
X_flat_adv = np.array(X_flat_adv, dtype="float32")

print(f"✅ Dataset shapes:")
print(f"   X_cnn: {X_cnn.shape}")
print(f"   X_flat_basic: {X_flat_basic.shape}")
print(f"   X_flat_adv: {X_flat_adv.shape}")
print(f"   y: {len(y)}")

le = LabelEncoder()
y = le.fit_transform(y)

print(f"   Classes: {le.classes_}")
print(f"   Class distribution: {np.bincount(y)}")

# ============================================================
# Model Builders - ENHANCED
# ============================================================
def build_mlp(input_dim, units):
    model = Sequential()
    model.add(Dense(units[0], activation="relu", input_dim=input_dim))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))

    for u in units[1:]:
        model.add(Dense(u, activation="relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.4))

    model.add(Dense(NUM_CLASSES, activation="softmax"))

    # ✨ IMPROVED: Better learning rate schedule
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-3,
        decay_steps=500,
        decay_rate=0.96
    )

    model.compile(
        optimizer=Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def build_cnn(input_shape, filters):
    """✨ IMPROVED: Larger kernels and deeper architecture"""
    model = Sequential()

    for i, f in enumerate(filters):
        kernel_size = 7 if i == 0 else 5 if i == 1 else 3
        model.add(Conv1D(
            f, kernel_size, padding="same", activation="relu",
            input_shape=input_shape if i == 0 else None
        ))
        model.add(BatchNormalization())
        model.add(Conv1D(f, kernel_size, padding="same", activation="relu"))
        model.add(BatchNormalization())
        model.add(MaxPooling1D(2))
        model.add(Dropout(0.3))

    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.4))
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(0.3))
    model.add(Dense(NUM_CLASSES, activation="softmax"))

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=3e-4,
        decay_steps=500,
        decay_rate=0.96
    )

    model.compile(
        optimizer=Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ============================================================
# Training with Cross-Validation - NEW
# ============================================================
def get_class_weights(y_train):
    """✨ NEW: Compute balanced class weights"""
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    return {i: w for i, w in enumerate(class_weights)}

def train_model_with_cv(X, y, model_builder, model_name, use_cnn=False):
    """✨ NEW: Train with k-fold cross-validation"""
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    cv_scores = []
    all_preds = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/{N_SPLITS}", end=" ")
        
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        model = model_builder()
        
        class_weights = get_class_weights(y_train)
        
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )
        
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weights,
            callbacks=[early_stop],
            verbose=0
        )
        
        preds = np.argmax(model.predict(X_val, verbose=0), axis=1)
        f1 = f1_score(y_val, preds, average='weighted')
        cv_scores.append(f1)
        all_preds.append((y_val, preds))
        print(f"F1: {f1:.4f}")
    
    mean_f1 = np.mean(cv_scores)
    std_f1 = np.std(cv_scores)
    
    print(f"  ✅ Mean F1: {mean_f1:.4f} ± {std_f1:.4f}\n")
    
    return mean_f1, std_f1, all_preds

# ============================================================
# Configs
# ============================================================
configs = {
    "1_layer": [[32], [64], [128], [256]],
    "2_layer": [[32,64], [64,128], [128,256], [256,512]],
    "3_layer": [[32,64,128], [64,128,256]],
}

svm_configs = [
    {"C": 1, "kernel": "rbf"},
    {"C": 3, "kernel": "rbf"},
    {"C": 6, "kernel": "rbf"},
    {"C": 10, "kernel": "rbf"},
]

# ============================================================
# Experiments - IMPROVED
# ============================================================
best_models = {}

print("\n" + "="*60)
print("LUNG SOUND CLASSIFICATION - IMPROVED PIPELINE")
print("="*60)

# Test with advanced features first
print("\n✨ Testing with ADVANCED features (MFCC + Mel + ZCR + Spectral + RMS)")
print("="*60)

X_to_use = X_flat_adv  # Use advanced features

for model_type in ["MLP", "CNN", "SVM", "KNN"]:
    print(f"\n🔬 {model_type}")
    print("-" * 40)

    if model_type == "SVM":
        best_f1 = 0
        best_cfg = None
        best_model = None
        
        for cfg in svm_configs:
            model = SVC(C=cfg["C"], kernel=cfg["kernel"], random_state=42)
            model.fit(X_to_use, y)
            
            # Use cross-validation for SVM too
            from sklearn.model_selection import cross_val_score
            cv_scores = cross_val_score(
                model, X_to_use, y, 
                cv=5, 
                scoring='f1_weighted'
            )
            mean_f1 = cv_scores.mean()
            print(f"  {cfg} → F1: {mean_f1:.4f}")
            
            if mean_f1 > best_f1:
                best_f1 = mean_f1
                best_cfg = cfg
                best_model = SVC(C=cfg["C"], kernel=cfg["kernel"], random_state=42)
        
        best_model.fit(X_to_use, y)
        best_models["SVM"] = (best_cfg, best_f1, best_model)
        
    elif model_type == "KNN":
        model = KNeighborsClassifier(n_neighbors=5)
        from sklearn.model_selection import cross_val_score
        cv_scores = cross_val_score(model, X_to_use, y, cv=5, scoring='f1_weighted')
        best_f1 = cv_scores.mean()
        
        print(f"  n_neighbors=5 → F1: {best_f1:.4f}")
        best_models["KNN"] = ({"n_neighbors": 5}, best_f1, model)
        
    elif model_type == "MLP":
        print("\n  Searching architectures...")
        best_f1 = 0
        best_cfg = None
        
        for depth, cfgs in configs.items():
            print(f"    {depth}:")
            for units in cfgs:
                mean_f1, std_f1, _ = train_model_with_cv(
                    X_to_use, y,
                    lambda: build_mlp(X_to_use.shape[1], units),
                    f"MLP-{units}",
                    use_cnn=False
                )
                
                if mean_f1 > best_f1:
                    best_f1 = mean_f1
                    best_cfg = units
        
        best_models["MLP"] = (best_cfg, best_f1, None)
        
    elif model_type == "CNN":
        print("\n  Searching architectures...")
        best_f1 = 0
        best_cfg = None
        
        for depth, cfgs in configs.items():
            print(f"    {depth}:")
            for units in cfgs:
                mean_f1, std_f1, _ = train_model_with_cv(
                    X_cnn, y,
                    lambda u=units: build_cnn(X_cnn.shape[1:], u),
                    f"CNN-{units}",
                    use_cnn=True
                )
                
                if mean_f1 > best_f1:
                    best_f1 = mean_f1
                    best_cfg = units
        
        best_models["CNN"] = (best_cfg, best_f1, None)

# ============================================================
# Final Summary
# ============================================================
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

results_summary = []
for name, (cfg, f1, model) in best_models.items():
    print(f"\n🏆 {name}")
    print(f"   Config: {cfg}")
    print(f"   Weighted F1: {f1:.4f}")
    results_summary.append((name, f1))

print("\n" + "="*60)
print("RANKING")
print("="*60)
for i, (name, f1) in enumerate(sorted(results_summary, key=lambda x: x[1], reverse=True), 1):
    print(f"{i}. {name:10s} → F1: {f1:.4f}")

best_overall = max(results_summary, key=lambda x: x[1])
print(f"\n🎯 Best Model: {best_overall[0]} with F1: {best_overall[1]:.4f}")


==================== CNN ====================

🧪 1_layer


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[32] → F1: 0.7999


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[64] → F1: 0.7949


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[128] → F1: 0.8119


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


KeyboardInterrupt: 